# Section 4-1: Chunking the Videos
*Notes:* This notebook divides videos into smaller, overlap-aware chunks for downstream processing (e.g., transcription, embedding). Ensure `ffprobe` is available on the cluster driver and that the workspace has access to the video files.

The detailed background of this code is in this blog:

https://medium.com/@junshan0/chunking-the-videos-b599ae25113a?sk=13819c9b5856e74c924ea83ab683fcce

*Note:* The blog includes examples for choosing chunk sizes and managing overlaps.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS video_ai.silver;

CREATE TABLE IF NOT EXISTS video_ai.silver.video_chunks (
  chunk_id     STRING,
  video_id     STRING,
  video_path   STRING,
  start_time   DOUBLE,   -- seconds
  end_time     DOUBLE,   -- seconds
  duration     DOUBLE,   -- seconds
  chunk_index  INT,
  created_at   TIMESTAMP
);

-- Comment: `video_chunks` stores metadata about each chunk. Use `chunk_index` to order chunks and
-- `start_time`/`end_time` for extracting the clip when needed. Run in a SQL cell.

In [ ]:
import subprocess, json

def get_duration_seconds(local_or_volume_path: str) -> float:
    """Return video duration in seconds using ffprobe."""
    # Runs ffprobe on the given local path. ffprobe must be available on the driver node.
    result = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json",
         "-show_format", local_or_volume_path],
        capture_output=True, text=True, check=True,
    )
    return float(json.loads(result.stdout)["format"]["duration"])

# Comment: If ffprobe is not available, consider using a library or a short containerized helper to compute durations.

In [ ]:
import uuid
from pyspark.sql import functions as F

CHUNK_SECONDS   = 300   # 5-minute chunks
OVERLAP_SECONDS = 15    # small overlap so boundary context isn't lost

def make_chunks(video_id, video_path, duration):
    rows, start, idx = [], 0.0, 0
    while start < duration:
        end = min(start + CHUNK_SECONDS, duration)
        rows.append((
            str(uuid.uuid4()),          # chunk_id
            video_id, video_path,
            float(start), float(end), float(end - start), idx,
        ))
        if end >= duration:
            break
        start = end - OVERLAP_SECONDS
        idx += 1
    return rows

# Read only files that have not been chunked yet
videos = spark.table("video_ai.bronze.video_files") \
    .filter(F.col("processing_status") == "INGESTED") \
    .select("file_name", "file_path").collect()

all_rows = []
for v in videos:
    video_id = v["file_name"]
    # Note: `get_duration_seconds` expects a local path or driver-accessible path
    duration = get_duration_seconds(v["file_path"])   # see Step 3 on path access
    all_rows += make_chunks(video_id, v["file_path"], duration)

columns = ["chunk_id", "video_id", "video_path",
           "start_time", "end_time", "duration", "chunk_index"]

chunks_df = (
    spark.createDataFrame(all_rows, columns)
    .withColumn("created_at", F.current_timestamp())
)

chunks_df.write.mode("append").saveAsTable("video_ai.silver.video_chunks")

# Comment: This approach calls ffprobe for each file; for large batches consider a parallelized helper or per-node processing.

In [ ]:
import os
import uuid
from pyspark.sql import functions as F

CHUNK_SECONDS   = 300   # 5-minute chunks
OVERLAP_SECONDS = 15    # small overlap so boundary context isn't lost

def make_chunks(video_id, video_path, duration):
    rows, start, idx = [], 0.0, 0
    while start < duration:
        end = min(start + CHUNK_SECONDS, duration)
        rows.append((
            str(uuid.uuid4()),          # chunk_id
            video_id, video_path,
            float(start), float(end), float(end - start), idx,
        ))
        if end >= duration:
            break
        start = end - OVERLAP_SECONDS
        idx += 1
    return rows

# Read only files that have not been chunked yet
videos = spark.table("video_ai.bronze.video_files") \
    .filter(F.col("processing_status") == "INGESTED") \
    .select("file_name", "file_path").collect()

# ffprobe cannot read S3 URIs directly — copy to a local Workspace path first
LOCAL_DIR = "/Workspace/Users/databricks_certifications@outlook.com/tmp_video"
os.makedirs(LOCAL_DIR, exist_ok=True)

all_rows = []
for v in videos:
    video_id = v["file_name"]
    s3_path = v["file_path"]
    local_path = os.path.join(LOCAL_DIR, video_id)
    # Copy from S3 (or workspace volume) to local driver path so ffprobe can access it
    dbutils.fs.cp(s3_path, f"file:{local_path}")
    duration = get_duration_seconds(local_path)
    all_rows += make_chunks(video_id, s3_path, duration)
    dbutils.fs.rm(f"file:{local_path}")

columns = ["chunk_id", "video_id", "video_path",
           "start_time", "end_time", "duration", "chunk_index"]

chunks_df = (
    spark.createDataFrame(all_rows, columns)
    .withColumn("created_at", F.current_timestamp())
)

chunks_df.write.mode("append").saveAsTable("video_ai.silver.video_chunks")

# Comment: Clean up local files promptly to avoid driver disk pressure. Consider using ephemeral workers or mounting S3.

In [ ]:
%sql
INSERT INTO video_ai.silver.video_chunks
SELECT
  uuid()                                            AS chunk_id,
  d.video_id,
  d.video_path,
  w.start_time,
  least(w.start_time + 300, d.duration)             AS end_time,
  least(w.start_time + 300, d.duration) - w.start_time AS duration,
  CAST(w.start_time / 285 AS INT)                   AS chunk_index,   -- 300s window - 15s overlap
  current_timestamp()                               AS created_at
FROM video_ai.silver.video_durations d
LATERAL VIEW explode(
  sequence(0D, d.duration, 285D)   -- step = chunk (300) - overlap (15)
) w AS start_time;

-- Comment: This SQL computes chunk windows using a sequence; adjust step/window as needed.

In [ ]:
%sql
SELECT video_id, count(*) AS chunk_count,
       min(start_time) AS first_start, max(end_time) AS last_end
FROM video_ai.silver.video_chunks
GROUP BY video_id
ORDER BY video_id;

-- Comment: Quick QA to verify expected chunk counts per video.

In [ ]:
%sql
UPDATE video_ai.bronze.video_files
SET processing_status = 'CHUNKED'
WHERE file_name IN (SELECT DISTINCT video_id FROM video_ai.silver.video_chunks);

-- Comment: Marks source files as chunked so downstream steps skip them. Run after successful chunk creation.